# EgoIndustrial Inference Demo

This notebook demonstrates how to run inference on local video files using the trained EgoIndustrial model.

## Prerequisites
- Trained model checkpoint (.ckpt)
- TensorRT engine (.engine) for fast inference
- Local video file(s) to analyze

## Setup

In [ ]:
# Install dependencies
%pip install -e ".[inference]" -q

In [ ]:
import os
import cv2
import torch
import numpy as np
from pathlib import Path
from tqdm import tqdm

from egoindustrial.inference.tensorrt_engine import TensorRTEngine
from egoindustrial.data.transforms import get_transforms

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

Set paths to your model engine and video file.

In [ ]:
# Configuration - UPDATE THESE PATHS
ENGINE_PATH = "outputs/model_int8.engine"  # or model_fp16.engine
VIDEO_PATH = "path/to/your/video.mp4"  # UPDATE THIS
OUTPUT_DIR = "outputs/inference"

# Inference settings
CLIP_LEN = 16
FRAME_STRIDE = 2
BATCH_SIZE = 1
CONFIDENCE_THRESHOLD = 0.3

# Class names (EPIC-KITCHENS)
EPIC_VERBS = [
    "take", "put", "open", "close", "wash", "cut", "pour", "mix", "shake",
    "sprinkle", "peel", "slice", "chop", "dice", "grate", "squeeze", "spread",
    "dip", "stir", "fold", "unwrap", "wrap", "scoop", "drop", "throw", "pick",
    "place", "move", "hold", "touch", "press", "push", "pull", "turn", "rotate",
    "flip", "shake", "tap", "knock", "hit", "bang", "scratch", "rub", "wipe",
    "clean", "dry", "soak", "rinse", "drain", "strain", "filter", "separate",
    "combine", "add", "remove", "replace", "adjust", "measure", "check", "taste",
    "smell", "look", "watch", "wait", "rest", "walk", "stand", "sit", "bend",
    "reach", "extend", "lift", "lower", "carry", "bring", "fetch", "get", "find",
]

EPIC_NOUNS = [
    "knife", "spoon", "fork", "cup", "bowl", "plate", "pan", "pot", "lid",
    "handle", "drawer", "door", "fridge", "oven", "microwave", "sink", "tap",
    "water", "oil", "salt", "pepper", "sugar", "flour", "egg", "milk", "butter",
    "cheese", "bread", "tomato", "onion", "garlic", "pepper", "carrot", "potato",
    "meat", "fish", "chicken", "beef", "pork", "rice", "pasta", "noodle", "sauce",
    "soup", "salad", "sandwich", "pizza", "cake", "cookie", "fruit", "vegetable",
    "hand", "finger", "thumb", "palm", "wrist", "arm", "elbow", "shoulder",
    "towel", "cloth", "sponge", "brush", "whisk", "ladle", "spatula", "tongs",
    "peeler", "grater", "opener", "scissors", "chopper", "blender", "mixer",
]

In [ ]:
# Load TensorRT engine
engine = TensorRTEngine(ENGINE_PATH)

# Get transforms (validation mode, no augmentation)
transforms = get_transforms({
    "clip_len": CLIP_LEN,
    "crop_size": 224,
    "resize_size": 256,
    "is_train": False,
    "model_type": "videomae",
})

## Video Preprocessing Function

In [ ]:
def load_and_preprocess_video(video_path: str, clip_len: int = 16, frame_stride: int = 2):
    """
    Load video and sample clips for inference.

    Args:
        video_path: Path to video file
        clip_len: Number of frames per clip
        frame_stride: Stride between sampled frames

    Returns:
        List of preprocessed clips [T, C, H, W] ready for batching
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"Video: {video_path}")
    print(f"  FPS: {fps:.1f}, Frames: {total_frames}, Resolution: {width}x{height}")

# Read all frames
frames = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frames.append(frame_rgb)
cap.release()

frames = np.array(frames)  # [T, H, W, C]
total_frames = len(frames)

# Sample clips with sliding window
clips = []
clip_frame_indices = []
required_frames = clip_len * frame_stride

for start_idx in range(0, total_frames - required_frames + 1, required_frames // 2):  # 50% overlap
    indices = np.arange(start_idx, start_idx + required_frames, frame_stride)
    if len(indices) < clip_len:
        break
    clip = frames[indices]  # [clip_len, H, W, C]
    
    # Convert to tensor and normalize
    clip_tensor = torch.from_numpy(clip).permute(3, 0, 1, 2).float() / 255.0  # [C, T, H, W]
    clip_tensor = transforms(clip_tensor)  # Apply transforms

    clips.append(clip_tensor)
    clip_frame_indices.append(indices)

if not clips:
    # If video is too short, pad the last frames
    clip = frames[-clip_len:]
    clip_tensor = torch.from_numpy(clip).permute(3, 0, 1, 2).float() / 255.0
    clip_tensor = transforms(clip_tensor)
    clips.append(clip_tensor)
    clip_frame_indices.append(np.arange(max(0, total_frames - clip_len), total_frames))

return clips, clip_frame_indices, fps

In [ ]:
# Test with a video
clips, clip_indices, fps = load_and_preprocess_video(VIDEO_PATH, CLIP_LEN, FRAME_STRIDE)
print(f"Loaded {len(clips)} clips from video")
print(f"Clip shape: {clips[0].shape}")  # Should be [C, T, H, W]

## Run Inference

In [ ]:
def run_inference(engine, clips, batch_size=1):
    """Run inference on clips."""
    results = []

    for i in range(0, len(clips), batch_size):
        batch_clips = clips[i:i+batch_size]
        
        # Stack batch
        batch = torch.stack(batch_clips)  # [B, C, T, H, W]
        batch_np = batch.numpy()
        
        # Run inference
        outputs = engine.infer(batch_np)
        
        # Process outputs
        verb_logits = outputs[0]   # [B, num_verb]
        noun_logits = outputs[1]   # [B, num_noun]
        action_logits = outputs[2] # [B, num_action]
        
        for b in range(len(batch_clips)):
            verb_probs = torch.softmax(torch.from_numpy(verb_logits[b]), dim=-1)
            noun_probs = torch.softmax(torch.from_numpy(noun_logits[b]), dim=-1)
            action_probs = torch.softmax(torch.from_numpy(action_logits[b]), dim=-1)
            
            verb_top5 = verb_probs.topk(5)
            noun_top5 = noun_probs.topk(5)
            action_top5 = action_probs.topk(5)
            
            results.append({
                "verb": {
                    "top1": verb_top5.indices[0].item(),
                    "top1_prob": verb_top5.values[0].item(),
                    "top5": [(EPIC_VERBS[i] if i < len(EPIC_VERBS) else f"verb_{i}", v.item()) 
                              for i, v in zip(verb_top5.indices, verb_top5.values)]
                },
                "noun": {
                    "top1": noun_top5.indices[0].item(),
                    "top1_prob": noun_top5.values[0].item(),
                    "top5": [(EPIC_NOUNS[i] if i < len(EPIC_NOUNS) else f"noun_{i}", v.item()) 
                              for i, v in zip(noun_top5.indices, noun_top5.values)]
                },
                "action": {
                    "top1": action_top5.indices[0].item(),
                    "top1_prob": action_top5.values[0].item(),
                    "top5": [(action_top5.indices[j].item(), v.item()) 
                              for j, v in zip(action_top5.indices, action_top5.values)]
                },
            })
    
return results

# Run inference
results = run_inference(engine, clips, batch_size=1)
print(f"Processed {len(results)} clips")

## Visualize Results

In [ ]:
def print_results(results, clip_indices, fps, top_k=3):
    """Print formatted inference results."""
    for i, (result, indices) in enumerate(zip(results, clip_indices)):
        start_time = indices[0] / fps
end_time = indices[-1] / fps
        
        print(f"\n{'='*60}")
print(f"Clip {i+1}: {start_time:.1f}s - {end_time:.1f}s")
print(f"{'='*60}")
        
print(f"VERB:")
for j, (label, prob) in enumerate(result['verb']['top5'][:3]):
print(f"  {j+1}. {label}: {prob:.3f}")
        
print(f"NOUN:")
for j, (label, prob) in enumerate(result['noun']['top5'][:3]):
print(f"  {j+1}. {label}: {prob:.3f}")
        
print(f"ACTION:")
for j, (label, prob) in enumerate(result['action']['top5'][:3]):
print(f"  {j+1}. {label}: {prob:.3f}")

print_results(results, clip_indices, fps)

## Create Annotated Output Video

In [ ]:
def create_annotated_video(
    video_path: str,
results: list,
clip_indices: list,
fps: float,
output_path: str,
top_k: int = 3,
):
    """Create annotated video with prediction overlays."""
cap = cv2.VideoCapture(video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
input_fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, input_fps, (width, height))

frame_idx = 0
clip_idx = 0
clip_frame_idx = 0

while True:
    ret, frame = cap.read()
if not ret:
    break
    
    # Update clip index
if clip_idx < len(clip_indices) and frame_idx > clip_indices[clip_idx][-1]:
clip_idx += 1
clip_frame_idx = 0
    
    # Draw predictions on frame
if clip_idx < len(results):
result = results[clip_idx]
        
        # Draw verb
verb_text = f"Verb: {results[clip_idx]['verb']['top5'][0][0]} ({results[clip_idx]['verb']['top5'][0][1]:.2f})"
cv2.putText(frame, verb_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        
        # Draw noun
noun_text = f"Noun: {results[clip_idx]['noun']['top5'][0][0]} ({results[clip_idx]['noun']['top5'][0][1]:.2f})"
cv2.putText(frame, noun_text, (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        
        # Draw action
action_text = f"Action: {results[clip_idx]['action']['top5'][0][0]} ({results[clip_idx]['action']['top5'][0][1]:.2f})"
cv2.putText(frame, action_text, (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        
        # Draw clip info
info_text = f"Clip: {clip_idx+1}/{len(results)} | Frame: {clip_frame_idx}"
cv2.putText(frame, info_text, (10, height - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
out.write(frame)
frame_idx += 1
clip_frame_idx += 1

cap.release()
out.release()
print(f"Annotated video saved to {output_path}")

In [ ]:
# Generate annotated video
OUTPUT_VIDEO = "outputs/inference/annotated_output.mp4"
os.makedirs(os.path.dirname(OUTPUT_VIDEO), exist_ok=True)

create_annotated_video(
    VIDEO_PATH,
results,
clip_indices,
fps,
OUTPUT_VIDEO,
)

## Batch Processing Multiple Videos

In [ ]:
def process_video_directory(input_dir: str, output_dir: str, engine, transforms, 
                            clip_len=16, frame_stride=2, batch_size=4):
    """Process all videos in a directory."""
input_path = Path(input_dir)
output_path = Path(output_dir)
output_path.mkdir(parents=True, exist_ok=True)

video_extensions = {'.mp4', '.avi', '.mov', '.mkv', '.MP4', '.AVI', '.MOV', '.MKV'}
video_files = [f for f in input_path.iterdir() if f.suffix in video_extensions]

print(f"Found {len(video_files)} videos to process")

for video_file in tqdm(video_files, desc="Processing videos"):
    try:
        # Load and preprocess
        clips, clip_indices, fps = load_and_preprocess_video(
            str(video_file), CLIP_LEN, FRAME_STRIDE
        )
        
        # Run inference
        results = run_inference(engine, clips, batch_size=4)
        
        # Print results
        print(f"\n{video_file.name}:")
        for i, result in enumerate(results):
            print(f"  Clip {i+1}: Verb={result['verb']['top5'][0][0]} ({result['verb']['top5'][0][1]:.2f}), "
                  f"Noun={result['noun']['top5'][0][0]} ({result['noun']['top5'][0][1]:.2f})")
        
        # Create annotated video
        output_video = output_path / f"annotated_{video_file.stem}.mp4"
        create_annotated_video(
            str(video_file),
results,
            [list(range(len(c))) for c in clips],  # simplified
30.0,  # default fps
str(output_video),
        )
        
    except Exception as e:
        print(f"Error processing {video_file}: {e}")

print("Done!")

In [ ]:
# Example: Process a directory of videos
# process_video_directory(
    # "path/to/videos",
    # "outputs/inference/videos",
    # engine,
    # transforms,
    # clip_len=CLIP_LEN,
    # frame_stride=FRAME_STRIDE,
    # batch_size=4,
    # )

## FastAPI Server Alternative

For production use, deploy the FastAPI server instead:

In [ ]:
# Terminal command (run in separate terminal):
# python -m egoindustrial.inference.server \
#     --engine outputs/model_int8.engine \
#     --host 0.0.0.0 --port 8000

# Then call from Python:
import requests
import json

# def call_api(video_clip):
#     """Call FastAPI inference server."""
#     url = "http://localhost:8000/infer"
#     payload = {"video": video_clip.tolist()}
#     response = requests.post(url, json=payload)
#     return response.json()

## Summary

This notebook demonstrates:
1. **Loading TensorRT engine** for fast inference
2. **Video preprocessing** with sliding window clip sampling
3. **Running inference** on video clips
4. **Visualizing results** with top-k predictions
5. **Creating annotated output videos** with prediction overlays
6. **Batch processing** multiple videos

## Next Steps
- Deploy FastAPI server for production
- Add tracking/temporal smoothing for action recognition
- Integrate with weak supervision UI for label verification
- Deploy to GCP Cloud Run or Modal for scaling